# SnapShop demo
Any photo → **YOLO** finds each garment → for each garment: **tags** (group + colour) and the **top-10 similar H&M products**.
(Output 3, "buy next", is switched off for now: `ENABLE_RECS` in `src/predict.py`.)
Run cell 1 once (loads the models, ~20 s), then any `snapshop(...)` cell.

In [ ]:
import sys
sys.path.append("../src")

import matplotlib.pyplot as plt
import torch
from PIL import Image
from ultralytics import YOLO

from predict import (ALPHA, ENABLE_RECS, IMG_DIR, YOLO_W, build_index, buy_next, detect, final_group, load_models,
                     load_photo, load_recommender, prepare, run_models, search, tags)

device = "cuda" if torch.cuda.is_available() else "cpu"
yolo = YOLO(str(YOLO_W))
tagger, designer = load_models(device)
rec = load_recommender(device) if ENABLE_RECS else None   # output 3 switched off for now (ENABLE_RECS in predict.py)
indexes = {}                                     # one search index per match setting, built when first used
print("recommender:", rec["kind"] if rec else "off")


def show_product(ax, aid, title):
    path = IMG_DIR / aid[:3] / f"{aid}.jpg"
    if path.exists():
        ax.imshow(Image.open(path))
    else:                                        # bought items outside our cleaned catalogue (e.g. bags)
        ax.text(0.5, 0.5, f"{aid}\n(no photo)", ha="center", va="center", fontsize=7)
    ax.set_title(title, fontsize=7)


def snapshop(path, match="balanced", k=10, use_yolo=True, group_hint=True, n_recs=12):
    per = 2 if rec else 1                        # rows per garment: similar (+ buy next if on)
    alpha = ALPHA[match]
    if match not in indexes:
        indexes[match] = build_index(alpha)
    index, meta = indexes[match]
    row_of = {a: i for i, a in enumerate(meta["ids"])}
    photo = load_photo(path)
    garments = detect(photo, yolo) if use_yolo else [(photo, "full photo (no detector)", None)]

    cols = (max(k, n_recs) if rec else k) + 2
    fig, axes = plt.subplots(per * len(garments), cols, figsize=(1.6 * cols, 3.2 * per * len(garments)), squeeze=False)
    for g_i, (crop, label, hint) in enumerate(garments):
        hint = hint if group_hint else None
        top = axes[per * g_i]
        out_g, out_c, colour_fp, design_fp = run_models(tagger, designer, prepare(crop), device)
        t = tags(out_g, out_c, 1)
        g, why = final_group(t["group"], hint)
        c, cc = t["colour"][0]
        top[0].imshow(photo)
        top[0].set_title("YOUR PHOTO", fontsize=9, weight="bold")
        top[1].imshow(crop)
        top[1].set_title(f"{label}\n{g.replace('Garment ', '')} ({why})\n{c} {cc:.0%}", fontsize=8, color="darkgreen")
        scores, idx = search(index, design_fp, colour_fp, alpha, k, meta, hint)
        for rank, (ax, s, j) in enumerate(zip(top[2:], scores, idx), start=1):
            show_product(ax, meta["ids"][j], f"#{rank} {meta['group'][j].replace('Garment ', '')}\n{meta['colour'][j]} ({s:.2f})")
        if not rec:
            continue
        bot = axes[per * g_i + 1]
        recs, seed = buy_next(rec, [meta["ids"][j] for j in idx], device, n_recs)
        bot[1].text(0.5, 0.5, f"BUY NEXT\n({rec['kind']})\nafter {seed or 'trending now'}", ha="center", va="center",
                    fontsize=8, weight="bold", color="darkred")
        for rank, (ax, aid) in enumerate(zip(bot[2:], recs), start=1):
            r = row_of.get(aid)
            show_product(ax, aid, f"{rank}. " + (f"{meta['group'][r].replace('Garment ', '')}\n{meta['colour'][r]}" if r is not None else "other"))
    for ax in axes.flat:
        ax.axis("off")
    fig.suptitle(f"SnapShop  |  {'similar + buy next' if rec else 'tags + similar'}  |  match = {match}  |  "
                 f"detector = {'YOLO' if use_yolo else 'off'}  |  group hint = {'on' if (use_yolo and group_hint) else 'off'}",
                 fontsize=12, weight="bold")
    plt.tight_layout()
    plt.show()

## ★ Demo photos (from Google): every image in `data/street_test/demo/`

In [ ]:
from pathlib import Path
for p in sorted(Path("../data/street_test/demo").glob("*")):
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
        print(p.name)
        snapshop(p)

## 7. Clean photos of a person wearing an outfit (top + bottom → one row per garment)

In [ ]:
snapshop("../data/street_test/person_stripes_skirt.jpg", group_hint=False)   # before the fix
for name in ["person_stripes_skirt", "person_tee_shorts", "person_shirt_trousers", "person_top_skirt"]:
    snapshop(f"../data/street_test/{name}.jpg")

## 1. Jeans on a red flowery bed (fooled the classifier without the detector)

In [ ]:
snapshop("../data/street_test/yolo_in/df2_000032.jpg", use_yolo=False)   # before: full photo
snapshop("../data/street_test/yolo_in/df2_000032.jpg")                   # after: YOLO crop

## 2. Street photo: woman in a café (several garments, busy background)

In [ ]:
snapshop("../data/street_test/yolo_in/df2_012321.jpg")

## 3. Person wearing a printed tee

In [ ]:
snapshop("../data/street_test/yolo_in/df2_000005.jpg")

## 4. A non-H&M bag (no bag class in DeepFashion2 → falls back to the full photo)

In [ ]:
snapshop("../data/street_test/test1.jpg")

## 5. Same photo, three match settings (design ↔ colour dial)

In [ ]:
for m in ["design", "balanced", "colour"]:
    snapshop("../data/street_test/yolo_in/df2_012081.jpg", match=m)

## 6. Your own photo: put it in `data/street_test/` and change the name

In [ ]:
snapshop("../data/street_test/test1.jpg")